

---

# 🤖 트랜스포머 기반 한국어 챗봇 모델 구현 및 트러블슈팅 보고서

## 1. 사전 학습된 임베딩 모델 로드 및 데이터 증강

### 🚩 문제점

Word2Vec 기반의 Data Augmentation을 위해 `Kyubyong/wordvectors`의 한국어 사전 학습 모델(`ko.bin`)을 로드하는 과정에서 `AttributeError` 및 `UnicodeDecodeError`가 발생했습니다. 이는 해당 모델이 과거 Gensim 3.x 환경에서 저장된 모델 파일(Pickle)인 반면, 현재 개발 환경은 최신 파이썬(3.12)과 Gensim 4.x 버전을 사용하고 있어 발생하는 C언어 빌드 및 객체 구조 호환성 문제였습니다.

### 💡 해결

현재 환경에서 구버전 라이브러리를 직접 설치하는 것은 빌드 충돌을 야기하므로, 우회 전략을 사용했습니다. 구버전(Gensim 3.8.3)이 설치된 별도의 가상환경을 구축하여 `ko.bin`을 로드한 뒤, 이를 버전과 무관하게 읽을 수 있는 일반 텍스트 포맷(`ko_vectors.txt`)으로 추출(`save_word2vec_format`)했습니다. 이후 메인 환경에서 `KeyedVectors.load_word2vec_format()`을 통해 성공적으로 로드하여 `lexical_sub` (어휘 대체) 기법으로 데이터를 3배로 증강했습니다.

### 🔜 다음 문제점

증강된 데이터로 모델을 훈련시키고 텍스트 생성을 테스트(Inference)했으나, 모델이 "붙어 붙어 붙어..."와 같이 동일한 단어만 무한 반복하며 BLEU 점수 0점을 기록하는 현상이 발생했습니다.

---

## 2. 평가(Inference) 모드 전환 누락에 따른 무한 반복

### 🚩 문제점

문장을 생성하는 디코딩 함수(`translate`) 내부에 평가 모드 전환 코드가 누락되어 있었습니다. 이로 인해 추론 시에도 드롭아웃(Dropout, 0.3)이 활성화되어 매 토큰을 예측할 때마다 모델의 가중치 일부가 무작위로 비활성화되었고, 그 결과 문맥을 이어가지 못하고 모델이 불안정한 예측(단일 단어 반복)을 지속하는 문제가 확인되었습니다.

### 💡 해결

`translate` 함수 상단에 `model.eval()`을 추가하여 드롭아웃을 비활성화하고, 메모리 누수 방지 및 추론 속도 향상을 위해 예측 루프 전체를 `with torch.no_grad():` 블록으로 감싸주어 추론 환경을 안정화했습니다.

### 🔜 다음 문제점

추론 환경을 수정한 뒤 무한 반복 현상은 해결되었고 정상적으로 종료 토큰(`<EOS>`)을 뱉어내기 시작했으나, 모든 질문에 대해 "해보세요"와 같이 단 한 마디의 단답만 남기고 즉시 생성을 종료해 버리는 새로운 문제가 관찰되었습니다.

---

## 3. 과소적합(Underfitting)으로 인한 단답형 생성

### 🚩 문제점

"해보세요" 단답 출력은 오류가 아니라, 데이터 파이프라인과 디코더 생성이 정상 작동하기 시작했다는 증거였습니다. 단지 기존에 설정한 3 Epoch라는 절대적인 학습량 부족(Underfitting)으로 인해, 모델이 문맥을 충분히 파악하지 못하고 데이터셋에서 가장 안전하고 만만한 빈출 단어 하나만 내뱉은 뒤 훈련을 종료해 버리는 '안전제일주의'적 성향을 보인 것입니다.

### 💡 해결

모델의 구조(`n_layers=2`, `dropout=0.3`, 가중치 공유 등)가 과적합 방지에 충분히 강력하게 설계되어 있음을 확인했습니다. 모델이 자연스럽게 문장을 생성할 수 있도록 조기 종료(Early Stopping) 제약 없이 모델이 충분한 문맥을 학습할 시간을 부여했습니다. 최종적으로 학습 에폭(Epoch)을 **30 Epoch**로 대폭 상향하여 최종 훈련을 성공적으로 진행했습니다.

---

## 4. 만능 답변 편향(Generic Response) 및 그리디 서치(Greedy Search)의 한계 극복
### 🚩 문제점
30 Epoch 훈련 후 모델이 완벽한 한국어 문장 형태를 스스로 생성하고 `<EOS>`를 정상적으로 출력하게 되었습니다. 그러나 "학교샘 좋아하는 사람 있나?", "가상화폐 쫄딱 망함" 등 다양한 맥락의 질문에 대해 오직 "저랑 이야기해 주는 게 좋을 것 같아요"라는 동일한 대답만 반복 출력하는 '모드 붕괴(Mode Collapse)' 현상이 발생했습니다. 이는 모델이 Loss를 최소화하기 위해 문맥과 상관없이 가장 안전하고 무난한 '만능 답변' 패턴에 편향되었고, 디코딩 시 무조건 확률 1위의 단어만 선택하는 그리디 서치(argmax) 방식을 사용했기 때문입니다.

### 💡 해결
챗봇의 응답 다양성(Diversity)과 맥락 대응력을 확보하기 위해, 추론(`translate`) 로직에서 1위 단어만 고집하는 `argmax` 연산을 제거했습니다. 대신 Logit 값에 온도 파라미터(Temperature $T=0.8$)를 적용하여 확률 분포를 조정한 뒤, `torch.multinomial`을 이용한 확률적 샘플링(Sampling) 기법을 도입했습니다. 이를 통해 모델이 문맥에 맞는 상위 n개의 단어들 중 다양한 선택지를 탐색하도록 유도하여 일원화된 단답형/만능형 답변 구조를 획기적으로 개선할 수 있었습니다.


## 5. 하이퍼파라미터 튜닝을 통한 심층 과적합(Overfitting) 및 모드 붕괴 해결

### 🚩 문제점

추론 단계에서 온도 스케일링(Temperature Scaling)과 확률적 샘플링(Multinomial Sampling)을 도입하여 모델의 창의성을 강제했음에도 불구하고, 여전히 100% 동일한 만능 답변("저랑 이야기해 주는 게 좋을 것 같아요")만 출력되는 치명적인 문제가 확인되었습니다. 이는 디코딩 로직의 문제를 넘어, **모델 자체가 30 Epoch라는 긴 학습 기간 동안 특정 '안전한 정답'에 완전히 과적합(Overfitting)되어 발생한 심층적인 모드 붕괴(Mode Collapse)** 현상이었습니다. 모델의 뇌 용량(`d_model=512`)이 데이터셋 규모에 비해 너무 커서, 문맥(Cross-Attention)을 이해하는 대신 Loss를 최소화하는 특정 문장 자체를 통째로 암기해 버린 것이 원인이었습니다.

### 💡 해결

단순 암기를 방지하고 모델이 언어의 범용적 패턴과 문맥을 추론하도록 강제하기 위해, 트랜스포머 모델의 핵심 하이퍼파라미터를 전면 재조정(Tuning)했습니다.

1. **모델 용량 다이어트 및 깊이 증가:** 임베딩 차원(`d_model`)을 512에서 **256**으로 절반으로 줄여 단순 암기 공간을 축소하는 대신, 인코더/디코더 층(`n_layers`)을 2에서 **4**로 늘려 복잡한 문맥 추론 능력을 강화했습니다.
2. **정규화(Regularization) 강화:** `dropout` 비율을 0.3에서 **0.4**로 상향하여, 학습 중 특정 가중치(만능 답변으로 향하는 뉴런)에 대한 의존도를 낮추고 더 다양한 단어 조합을 탐색하도록 유도했습니다.
3. **최적 학습 구간(Goldilocks Zone) 설정:** 모델이 과적합의 늪에 빠지기 전인 **15 Epoch**로 학습 횟수를 단축하고, 망가진 가중치를 초기화하여 새로운 구조로 재학습을 진행했습니다. 추가로 추론 함수에는 **Top-K 샘플링**을 적용하여 안정적이면서도 다채로운 대화를 생성할 수 있도록 파이프라인을 최종 완성했습니다.




## 6. N-gram 무한 루프 탈출 및 타겟 데이터(Target Data) 증강 부작용 교정

### 🚩 문제점

하이퍼파라미터 튜닝 후, 모델의 크로스 어텐션(Cross-Attention)이 정상화되어 질문의 핵심 키워드("연락", "망함", "사랑" 등)를 정확히 캐치하기 시작했습니다. 그러나 텍스트 생성 과정에서 "그렇 마음 그렇 마음..." 또는 "두 두 두..."와 같이 생성 모델의 고질적 병폐인 **N-gram 무한 반복 루프**에 빠지는 현상이 관찰되었습니다. 이를 억제하기 위해 추론 과정에 반복 페널티를 부여하자, 이번에는 조사와 어미가 완전히 파괴되고 명사만 나열되는 **단어 샐러드(Word Salad)** 현상("연락 생각 감기 큰 쉽길...")이 발생했습니다.

### 💡 해결

이러한 복합적인 생성 오류를 해결하기 위해 디코딩 로직과 데이터 파이프라인 양측을 전면 수정했습니다.

1. **추론(Decoding) 고도화:** 고지식한 Greedy 방식이나 단순 Top-K 방식을 버리고, 누적 확률 기반의 **Top-p (Nucleus) 샘플링**을 도입했습니다. 이와 함께 이전에 생성한 모든 토큰의 확률을 동적으로 깎는 **글로벌 반복 페널티(Dynamic Global Repetition Penalty)**를 적용하여 기계적인 단어 반복 루프를 완벽하게 끊어냈습니다.
2. **데이터 파이프라인 정규화 (근본 원인 해결):** 단어 샐러드 현상의 진짜 원인은 프로젝트 초기(Phase 1)에 적용했던 **답변(Target) 데이터의 Word2Vec 어휘 대체 증강**에 있음을 규명했습니다. 한국어 문법 특성상 타겟 문장의 단어를 임의로 대체하면 조사와 어미의 호응이 파괴되어, 모델이 이 망가진 문법을 정답으로 학습하게 됩니다. 이를 해결하기 위해 데이터 증강 파이프라인을 분리하여 **질문(Source) 데이터만 증강하고 답변(Target) 문장은 순수 원본을 유지**하도록 수정했습니다. 무결해진 타겟 데이터로 모델을 재학습시킴으로써, 마침내 문맥 파악과 문법적 유창성(Fluency)을 모두 갖춘 텍스트 생성을 달성할 수 있었습니다.



In [1]:
!mkdir -p ~/work/transformer_chatbot/data/spa-eng

In [ ]:
# !sudo apt-get install g++ openjdk-8-jdk
# !sudo apt-get install curl

# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# !pip install konlpy

In [58]:
!python3 -m pip install --upgrade pip
!python3 -m pip install konlpy # Python 3.x
# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh) # MeCab 설치하기

In [2]:
# from konlpy.tag import Mecab
#
!pip install python-mecab-ko
!pip install konlpy
from mecab import MeCab
mecab = MeCab()

print(mecab.morphs("이제 에러 없이 잘 돌아가나요?"))

try:
    # 기본 호출
    # mecab = Mecab()
    print("Mecab 로드 성공:", mecab.morphs("아버지가방에들어가신다"))
except Exception as e:
    print("에러 발생:", e)

['이제', '에러', '없이', '잘', '돌아가', '나요', '?']
Mecab 로드 성공: ['아버지', '가', '방', '에', '들어가', '신다']


In [3]:
!pip install sentencepiece nltk

In [4]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.7.1+cu118


In [5]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

print("슝=3")

슝=3


In [10]:
!wget https://github.com/songys/Chatbot_data/raw/refs/heads/master/ChatbotData.csv


df = pd.read_csv("./ChatbotData.csv")#, names =["eng"])

df

questions = df["Q"]
answers = df["A"]
# answers = df["A"]

answers

--2026-03-11 03:03:07--  https://github.com/songys/Chatbot_data/raw/refs/heads/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv [following]
--2026-03-11 03:03:07--  https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.6’

ChatbotData.csv.6   100%[===================>] 868.99K  --.-KB/s    in 0.1s    

2026-03-11 03:03:08 (5.72 MB/s) - ‘ChatbotData.csv.6’ saved [889842/889842]



0                      하루가 또 가네요.
1                       위로해 드립니다.
2                     여행은 언제나 좋죠.
3                     여행은 언제나 좋죠.
4                      눈살이 찌푸려지죠.
                   ...           
11818          티가 나니까 눈치가 보이는 거죠!
11819               훔쳐보는 거 티나나봐요.
11820                      설렜겠어요.
11821    잘 헤어질 수 있는 사이 여부인 거 같아요.
11822          도피성 결혼은 하지 않길 바라요.
Name: A, Length: 11823, dtype: object

In [11]:
extracted_folder = "./spa-eng"
# file_path = os.path.join(extracted_folder, "spa.txt")

# with open(file_path, "r") as f:
    # spa_eng_sentences = f.read().splitlines()

# spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(df)
print("Example:", total_sentence_count)

for sen in df[0:100][::20]:
    print(">>", sen)

Example: 11823
>> Q
>> A
>> label


In [12]:
import os
import re
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-zA-Z?.!,가-힣ㄱ-ㅎ-ㅣ]+", " ", sentence)
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

In [13]:

questions = questions.apply(preprocess_sentence)
answers = answers.apply(lambda x: preprocess_sentence(x))

print('슝=3')
questions

슝=3


0                           시 땡 !
1                      지망 학교 떨어졌어
2                     박 일 놀러가고 싶다
3                  박 일 정도 놀러가고 싶다
4                         ppl 심하네
                   ...           
11818             훔쳐보는 것도 눈치 보임 .
11819             훔쳐보는 것도 눈치 보임 .
11820                흑기사 해주는 짝남 .
11821    힘든 연애 좋은 연애라는게 무슨 차이일까 ?
11822                  힘들어서 결혼할까봐
Name: Q, Length: 11823, dtype: object

In [14]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_kor_q_sentences = questions[:-test_sentence_count]
test_kor_q_sentences = questions[-test_sentence_count:]
train_kor_a_sentences = answers[:-test_sentence_count]
test_kor_a_sentences = answers[-test_sentence_count:]
print("Train Example:", len(train_kor_q_sentences))
for sen in train_kor_q_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_kor_q_sentences))
for sen in test_kor_q_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  59


Train Example: 11764
>> 시 땡 !
>> 가스비 비싼데 감기 걸리겠어
>> 간만에 떨리니까 좋더라
>> 감정컨트롤을 못하겠어
>> 개강룩 입어볼까


Test Example: 59
>> 학교샘 좋아하는 사람 있나 ?
>> 헤어지고 나서 알았어 사랑했다는 걸
>> 혼자가 편하다는 짝남에게 먼저 대쉬해버림 .


In [15]:

def tokenize_kor(text):
    # Mecab으로 형태소 분석 후 공백으로 연결된 문자열 반환
    return " ".join(mecab.morphs(text))

In [16]:
def split_q_a_sentences(qs_d, as_d):
    q_sentences = []
    a_sentences = []
    len_q = len(qs_d)
    for  i in tqdm(range(len_q)):
        # eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        q= qs_d[i]
        a = as_d[i]
        
        src_preprocessed = tokenize_kor(q)
        # src_ids = self.encoder_tokenizer.encode(src_preprocessed)
        q_sentences.append(tokenize_kor(q))
        a_sentences.append(tokenize_kor(a))
    return q_sentences, a_sentences

print('슝=3')

슝=3


In [17]:
train_q_sentences, train_a_sentences = split_q_a_sentences(train_kor_q_sentences, train_kor_a_sentences)
print(len(train_q_sentences))
print(train_q_sentences[0])
print('\n')
print(len(train_a_sentences))
print(train_a_sentences[0])

  0%|          | 0/11764 [00:00<?, ?it/s]

11764
시 땡 !


11764
하루 가 또 가 네요 .


In [18]:
# print(test_kor_q_sentences)
test_kor_q_sentences = test_kor_q_sentences.reset_index(drop=True)
test_kor_a_sentences = test_kor_a_sentences.reset_index(drop=True)
test_q_sentences, test_a_sentences = split_q_a_sentences(test_kor_q_sentences, test_kor_a_sentences)
print(len(test_q_sentences))
print(test_q_sentences[0])
print('\n')
print(len(test_a_sentences))
print(test_a_sentences[0])

  0%|          | 0/59 [00:00<?, ?it/s]

59
학교 샘 좋 아 하 는 사람 있 나 ?


59
존경 의 의미 라고 생각 하 세요 .


In [19]:
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="kor",
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d'\
        % (file, model, vocab_size) + \
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'\
        % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer

print("슝=3")

슝=3


In [20]:
VOCAB_SIZE = 10000
tokenizer = generate_tokenizer(train_kor_q_sentences + train_kor_a_sentences, VOCAB_SIZE, 'spa-eng')
tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=././spa-eng_corpus.txt --model_prefix=spa-eng_spm --vocab_size=10000--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ././spa-eng_corpus.txt
  input_format: 
  model_prefix: spa-eng_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all

True

In [161]:
# 토크나이저에서 PAD와 EOS의 ID 값을 가져옵니다. (기본적으로 PAD는 0, EOS는 2)
pad_id = 0 # SentencePiece 설정에 따라 tokenizer.pad_id()를 써도 됩니다.
eos_id = tokenizer.eos_id()

def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)
    # 2. 리스트가 비어있지 않고, 맨 마지막 토큰이 EOS(2)인 경우 검사
        if len(tokens) > 1 and tokens[-1] == eos_id:
            # 맨 끝의 EOS 토큰을 잠시 뽑아냅니다. (리스트에서 제거됨)
            eos_token = tokens.pop()
            
            # 그 앞에 있는 토큰이 PAD(0)일 경우, PAD가 안 나올 때까지 계속 뽑아서 버립니다.
            while len(tokens) > 0 and tokens[-1] == pad_id:
                tokens.pop()
                
            # PAD 청소가 끝난 깔끔한 문장 끝에 다시 EOS를 붙여줍니다.
            tokens.append(eos_token)
        corpus.append(tokens)
    
    return corpus

print('슝=3')

슝=3


In [162]:
q_corpus = make_corpus(train_kor_q_sentences, tokenizer)
a_corpus = make_corpus(train_kor_a_sentences, tokenizer)

100%|██████████| 11764/11764 [00:00<00:00, 151885.39it/s]


In [163]:
print(train_kor_q_sentences[0])
print(q_corpus[0])
print('\n')
print(train_kor_a_sentences[0])
print(a_corpus[0])

시 땡 !
[1, 1051, 4, 9862, 41, 2]


하루가 또 가네요 .
[1, 305, 6, 110, 104, 22, 2]


In [164]:
train_kor_a_sentences

print("1. 데이터 타입 정제 및 결측치 처리 중...")
from tqdm import tqdm

# (가정) 원본 데이터가 들어있는 리스트
# que_corpus = ["질문 문장 1", "질문 문장 2", ...]
# ans_corpus = ["답변 문장 1", "답변 문장 2", ...]

print("1. 질문(Question) 데이터 Augmentation 진행 중...")
# 원본 que_corpus를 변형하여 aug_que_corpus 생성
aug_que = [lexical_sub_mecab(q, wv) for q in tqdm(train_kor_q_sentences)]
# print("aug_que_corpus", aug_que_corpus.head())
print("2. 답변(Answer) 데이터 Augmentation 진행 중...")
# 원본 ans_corpus를 변형하여 aug_ans_corpus 생성
aug_ans = [lexical_sub_mecab(a, wv) for a in tqdm(train_kor_a_sentences)]

aug_que_corpus = make_corpus(aug_que, tokenizer)
aug_ans_corpus = make_corpus(aug_ans, tokenizer)
print("\n3. 병렬 데이터셋 구성 중...")
# ------------------------------------------------------------------
# [데이터 병합 로직]
# 1. 원본 que_corpus + 원본 ans_corpus (1배)
# 2. Augmentation된 que_corpus + 원본 ans_corpus (2배)
# 3. 원본 que_corpus + Augmentation된 ans_corpus (3배)
# ------------------------------------------------------------------

final_que_corpus = q_corpus + aug_que_corpus + q_corpus
final_ans_corpus = a_corpus + a_corpus + aug_ans_corpus

# 결과 확인
print("\n✅ 데이터 증강 및 병합 완료!")
print(f"▶ 원본 데이터 개수: {len(q_corpus)} 쌍")
print(f"▶ 최종 데이터 개수: {len(final_que_corpus)} 쌍 (정확히 3배!)")

# 데이터가 제대로 병렬을 이루었는지 일부 출력해서 확인해보기
print("\n[샘플 데이터 확인]")
for i in [0, len(q_corpus), len(q_corpus)*2]:
    print("-" * 40)
    if i == 0:
        print("(원본 질문 + 원본 답변)")
    elif i == len(q_corpus):
        print("(Aug 질문 + 원본 답변)")
    else:
        print("(원본 질문 + Aug 답변)")
    
    print(f"Q: {final_que_corpus[i]}")
    print(f"A: {final_ans_corpus[i]}")



1. 데이터 타입 정제 및 결측치 처리 중...
1. 질문(Question) 데이터 Augmentation 진행 중...


100%|██████████| 11764/11764 [00:18<00:00, 628.15it/s]


2. 답변(Answer) 데이터 Augmentation 진행 중...


100%|██████████| 11764/11764 [00:00<00:00, 143362.93it/s]


3. 병렬 데이터셋 구성 중...

✅ 데이터 증강 및 병합 완료!
▶ 원본 데이터 개수: 11764 쌍
▶ 최종 데이터 개수: 35292 쌍 (정확히 3배!)

[샘플 데이터 확인]
----------------------------------------
(원본 질문 + 원본 답변)
Q: [1, 1051, 4, 9862, 41, 2]
A: [1, 305, 6, 110, 104, 22, 2]
----------------------------------------
(Aug 질문 + 원본 답변)
Q: [1, 1051, 6, 4, 9862, 41, 2]
A: [1, 305, 6, 110, 104, 22, 2]
----------------------------------------
(원본 질문 + Aug 답변)
Q: [1, 1051, 4, 9862, 41, 2]
A: [1, 305, 6, 110, 612, 426, 22, 2]


In [165]:
# 1. 훈련 데이터의 첫 번째 정답(Target) 문장 토큰 확인
sample_tgt_tokens = final_ans_corpus[14] # (변수명은 실제 훈련에 쓰신 데이터명으로 맞춰주세요)

print("1. 타겟 토큰 리스트:")
print(sample_tgt_tokens)

print("\n2. 디코딩된 타겟 문장:")
        # tokens = tokenizer.encode_as_ids(sentence)
print(tokenizer.decode_ids(sample_tgt_tokens))

print(f"\n3. Tokenizer의 EOS ID: {tokenizer.eos_id()}")

1. 타겟 토큰 리스트:
[1, 301, 14, 108, 3977, 13, 2]

2. 디코딩된 타겟 문장:
돈은 다시 들어올 거예요

3. Tokenizer의 EOS ID: 2


In [ ]:
final_que_corpus

In [167]:
train_kor_q_sentences.head()

0             시 땡 !
1        지망 학교 떨어졌어
2       박 일 놀러가고 싶다
3    박 일 정도 놀러가고 싶다
4           ppl 심하네
Name: Q, dtype: object

In [168]:
final_que_corpus[0]

[1, 1051, 4, 9862, 41, 2]

In [169]:
# q_corpus = make_corpus(final_que_corpus, tokenizer)
# a_corpus = make_corpus(final_ans_corpus, tokenizer)

In [170]:
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(final_que_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(final_ans_corpus, max_len=MAX_LEN, pad_value=0)

print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]
print("슝=3")

print(enc_ndarray[0])
print(enc_ndarray[1])
print(enc_ndarray[2])
print(final_ans_corpus[0])
print(dec_ndarray[1])

torch.Size([35292, 50])
torch.Size([35292, 50])
슝=3
tensor([   1, 1051,    4, 9862,   41,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0])
tensor([   1, 7483,  762, 1506,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0])
tensor([   1, 3314,   99, 2570,   98,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0, 

In [171]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print("슝=3")

슝=3


In [172]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


In [173]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


In [174]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


In [175]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


In [176]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


In [177]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


In [178]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)  # 필요 시 입력에 dropout 적용 가능

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns

# 사용 예시: Encoder 인스턴스 생성 후 forward 호출
# encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
# out, enc_attns = encoder(x, mask)
print("슝=3")

슝=3


In [179]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [180]:
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [215]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=4,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.4,
    shared_fc=True,
    shared_emb=True)

transformer = transformer.to(device)

d_model = 256

print("슝=3")

슝=3


In [216]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


In [217]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)
# 스케줄러 & Early Stopping 객체 생성
lr_scheduler = CustomSchedule(d_model=512, warmup_steps=4000, optimizer=optimizer)
early_stopping = EarlyStopping(patience=5, path='best_chatbot_model.pth')
print("슝=3")

슝=3


In [218]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


In [219]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [220]:
import torch

class CustomSchedule:
    def __init__(self, d_model, warmup_steps=4000, optimizer=None):
        super(CustomSchedule, self).__init__()
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.optimizer = optimizer
        self.step_num = 0
        
    def step(self):
        # 매 배치(스텝)마다 호출되어 학습률을 업데이트합니다.
        self.step_num += 1
        lr = self.learning_rate()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
    def learning_rate(self):
        # 트랜스포머 논문의 학습률 계산 공식
        step = self.step_num
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        
        return (self.d_model ** -0.5) * min(arg1, arg2)

In [222]:
print("TEST")

TEST


In [226]:
%%time
# ---------------------------------------------------------
# 1. 초기화 셋업
# ---------------------------------------------------------
EPOCHS = 35 # 넉넉하게 50으로 설정해도 Early Stopping이 알아서 멈춰줍니다!

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    for batch, (src, tgt) in enumerate(train_dataloader):
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)
        
        # loss.backward()
        # !! 핵심 !! 가중치 업데이트 후 스케줄러의 step()을 호출하여 학습률 조정
        # optimizer.step()
        # lr_scheduler.step()
        
        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    # avg_train_loss = total_train_loss / len(train_dataloader)
    # print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}")

    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")

100%|██████████| 552/552 [01:57<00:00,  4.70it/s, Batch Loss=646.0135]


Epoch 1, Loss: 707.3286


100%|██████████| 552/552 [01:57<00:00,  4.72it/s, Batch Loss=593.2009]


Epoch 2, Loss: 648.3791


100%|██████████| 552/552 [01:57<00:00,  4.72it/s, Batch Loss=538.5913]


...

Epoch 33, Loss: 102.8787


100%|██████████| 552/552 [01:57<00:00,  4.71it/s, Batch Loss=98.0412] 


Epoch 34, Loss: 99.4272


100%|██████████| 552/552 [01:57<00:00,  4.71it/s, Batch Loss=94.1545] 

Epoch 35, Loss: 96.3018
CPU times: user 1h 12min 51s, sys: 12.9 s, total: 1h 13min 4s
Wall time: 1h 8min 16s


In [ ]:
import numpy as np

class EarlyStopping:
    def __init__(self, patience=5, delta=0.001, path='best_transformer.pth'):
        """
        patience: 검증 손실이 개선되지 않아도 참아주는 에폭 수
        delta: 개선되었다고 판단할 최소 변화량
        path: 가장 좋은 모델이 저장될 파일 경로
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        # 첫 번째 에폭일 때
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            
        # 손실이 이전 최고 기록보다 더 떨어지지 않거나 오히려 올랐을 때 (과적합 시작 징후)
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f'⚠️ EarlyStopping 카운트: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                
        # 손실이 개선되었을 때 (새로운 최고 기록 갱신)
        else:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """검증 손실이 감소하면 모델 가중치를 저장합니다."""
        print(f'✅ Validation Loss가 감소했습니다. 최고 성능 모델을 저장합니다: {self.path}')
        torch.save(model.state_dict(), self.path)

In [ ]:
# 아래 두 문장을 바꿔가며 테스트 해보세요
reference = "많 은 자연어 처리 연구자 들 이 트랜스포머 를 선호 한다".split()
candidate = "적 은 자연어 학 개발자 들 가 트랜스포머 을 선호 한다 요".split()

print("원문:", reference)
print("번역문:", candidate)
print("BLEU Score:", sentence_bleu([reference], candidate))

In [81]:
print("1-gram:", sentence_bleu([reference], candidate, weights=[1, 0, 0, 0]))
print("2-gram:", sentence_bleu([reference], candidate, weights=[0, 1, 0, 0]))
print("3-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 1, 0]))
print("4-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 0, 1]))

1-gram: 0.5
2-gram: 0.18181818181818182
3-gram: 2.2250738585072626e-308
4-gram: 2.2250738585072626e-308


In [82]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)  # smoothing_function 적용

print("BLEU-1:", calculate_bleu(reference, candidate, weights=[1, 0, 0, 0]))
print("BLEU-2:", calculate_bleu(reference, candidate, weights=[0, 1, 0, 0]))
print("BLEU-3:", calculate_bleu(reference, candidate, weights=[0, 0, 1, 0]))
print("BLEU-4:", calculate_bleu(reference, candidate, weights=[0, 0, 0, 1]))

print("\nBLEU-Total:", calculate_bleu(reference, candidate))

BLEU-1: 0.5
BLEU-2: 0.18181818181818182
BLEU-3: 0.010000000000000004
BLEU-4: 0.011111111111111112

BLEU-Total: 0.05637560315259291


In [244]:
import torch
import torch.nn.functional as F

def translate(tokens, model, src_tokenizer, tgt_tokenizer):
    # tokens: 입력 토큰 리스트
    # MAX_LEN: 최대 길이 (전역 변수 혹은 상수)
    # device: 모델과 데이터가 위치한 디바이스
    model.eval()
    
    # tokens 길이가 MAX_LEN보다 크면 자르고, 작으면 0으로 패딩
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
    else:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    # 배치 차원을 추가하여 텐서로 변환 (shape: [1, MAX_LEN])
    padded_tokens = torch.tensor([tokens], dtype=torch.long, device=device)

    ids = []
    # 디코더의 첫 입력은 BOS 토큰 (배치 차원 추가)
    output = torch.tensor([[tgt_tokenizer.bos_id()]], dtype=torch.long, device=device)
    with torch.no_grad():
        for i in range(MAX_LEN):
            enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(padded_tokens, output)
            predictions, _, _, _ = model(padded_tokens, output, enc_padding_mask, combined_mask, dec_padding_mask)

            next_word_logits = predictions[0, -1].clone()

            # 🌟 1. 무자비한 반복 페널티 (Dynamic Penalty)
            for past_id in set(ids):
                if next_word_logits[past_id] > 0:
                    next_word_logits[past_id] /= 2.0  # 양수면 확률을 반토막 냄
                else:
                    next_word_logits[past_id] *= 2.0  # 음수면 두 배로 더 낮춤
                next_word_logits[past_id] -= 5.0      # 확인 사살로 5점을 추가로 깎음

            # 🌟 2. Top-p (Nucleus) Sampling 적용
            temperature = 0.7  # 0.7 정도가 가장 대화가 자연스럽습니다.
            top_p = 0.9        # 누적 확률 90% 안에 드는 단어들만 후보로 삼습니다.
            
            scaled_logits = next_word_logits / temperature
            probs = F.softmax(scaled_logits, dim=-1)

            # 누적 확률 계산을 위해 내림차순 정렬
            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # top_p를 초과하는 확률을 가진 토큰들의 인덱스 찾기
            sorted_indices_to_remove = cumulative_probs > top_p
            
            # 첫 번째 토큰은 무조건 살리기 위해 한 칸씩 오른쪽으로 밀어줌
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            # 컷오프된 토큰들의 확률을 0(-무한대)으로 만듦
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            scaled_logits[indices_to_remove] = -1e9

            # 다시 Softmax를 취해 다트 던지기
            filtered_probs = F.softmax(scaled_logits, dim=-1)
            predicted_id = torch.multinomial(filtered_probs, num_samples=1).item()

            if tgt_tokenizer.eos_id() == predicted_id:
                break
                
            ids.append(predicted_id)
            new_token = torch.tensor([[predicted_id]], dtype=torch.long, device=device)
            output = torch.cat([output, new_token], dim=1)
    result = tgt_tokenizer.decode_ids(ids)
    return result

print("슝=3")

슝=3


In [245]:
def eval_bleu_single(model, src_sentence, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    src_tokens = src_tokenizer.encode_as_ids(src_sentence)
    tgt_tokens = tgt_tokenizer.encode_as_ids(tgt_sentence)

    if (len(src_tokens) > MAX_LEN): return None
    if (len(tgt_tokens) > MAX_LEN): return None

    reference = tgt_sentence.split()
    candidate = translate(src_tokens, model, src_tokenizer, tgt_tokenizer).split()

    score = sentence_bleu([reference], candidate,
                          smoothing_function=SmoothingFunction().method1)

    if verbose:
        print("Source Sentence: ", src_sentence)
        print("Model Prediction: ", candidate)
        print("Real: ", reference)
        print("Score: %lf\n" % score)

    return score

print('슝=3')

슝=3


In [246]:
# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 0

eval_bleu_single(transformer,
                 test_kor_q_sentences[test_idx],
                 test_kor_a_sentences[test_idx],
                 tokenizer,
                 tokenizer)


# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 1

eval_bleu_single(transformer,
                 test_kor_q_sentences[test_idx],
                 test_kor_a_sentences[test_idx],
                 tokenizer,
                 tokenizer)

# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 2

eval_bleu_single(transformer,
                 test_kor_q_sentences[test_idx],
                 test_kor_a_sentences[test_idx],
                 tokenizer,
                 tokenizer)
test_idx = 16

eval_bleu_single(transformer,
                 train_kor_q_sentences[test_idx],
                 train_kor_a_sentences[test_idx],
                 tokenizer,
                 tokenizer)


test_idx = 10016

eval_bleu_single(transformer,
                 train_kor_q_sentences[test_idx],
                 train_kor_a_sentences[test_idx],
                 tokenizer,
                 tokenizer)


Source Sentence:  학교샘 좋아하는 사람 있나 ?
Model Prediction:  ['마음', '가', '꾸준히', '부모님', '것', '하나', '더', '사랑할', '생각', '그래요']
Real:  ['존경의', '의미라고', '생각하세요', '.']
Score: 0.000000

Source Sentence:  학교에 심남 있는데 연락해볼까 ?
Model Prediction:  ['연락', '생각', '감기', '큰', '쉽길', '바랍니다']
Real:  ['호감을', '어느', '정도', '표현해보는', '것도', '좋을', '것', '같아요', '.']
Score: 0.000000

Source Sentence:  학교에 좋아하는 여자애의 관심을 어떻게 끌지 ?
Model Prediction:  ['그분', '없어']
Real:  ['우선', '자신이', '멋진', '사람이', '되어', '보세요', '.']
Score: 0.000000

Source Sentence:  가상화폐 쫄딱 망함
Model Prediction:  ['어서', '어디', '오는', '하루', '더', '거예요']
Real:  ['어서', '잊고', '새출발', '하세요', '.']
Score: 0.040825

Source Sentence:  사랑해 보고싶어
Model Prediction:  ['얼', '사랑', '두', '부드', '달력', '많이', '자기', '저', '분위기', '힘들', '것예요']
Real:  ['얼른', '만나러가세요', '.']
Score: 0.000000



0

In [231]:
def eval_bleu(model, src_sentences, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    total_score = 0.0
    sample_size = len(src_sentences)

    for idx in tqdm(range(sample_size)):
        score = eval_bleu_single(model, src_sentences[idx], tgt_sentence[idx], src_tokenizer, tgt_tokenizer, verbose)
        if not score: continue

        total_score += score

    print("Num of Sample:", sample_size)
    print("Total Score:", total_score / sample_size)

print("슝=3")

슝=3


In [80]:
eval_bleu(transformer, test_kor_q_sentences, test_kor_a_sentences, tokenizer, tokenizer, verbose=False)

100%|██████████| 59/59 [00:18<00:00,  3.25it/s]

Num of Sample: 59
Total Score: 0.0008894098421667484


In [81]:
def beam_search_decoder(prob, beam_size):
    sequences = [[[], 1.0]]  # 생성된 문장과 점수를 저장

    for tok in prob:
        all_candidates = []

        for seq, score in sequences:
            for idx, p in enumerate(tok): # 각 단어의 확률을 총점에 누적 곱
                candidate = [seq + [idx], score * -math.log(-(p-1))]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates,
                         key=lambda tup:tup[1],
                         reverse=True) # 총점 순 정렬
        sequences = ordered[:beam_size] # Beam Size에 해당하는 문장만 저장

    return sequences

print("슝=3")

슝=3


In [82]:
vocab = {
    0: "<pad>",
    1: "까요?",
    2: "커피",
    3: "마셔",
    4: "가져",
    5: "될",
    6: "를",
    7: "한",
    8: "잔",
    9: "도",
}

prob_seq = [[0.01, 0.01, 0.60, 0.32, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.75, 0.01, 0.01, 0.17],
            [0.01, 0.01, 0.01, 0.35, 0.48, 0.10, 0.01, 0.01, 0.01, 0.01],
            [0.24, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.68],
            [0.01, 0.01, 0.12, 0.01, 0.01, 0.80, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.81, 0.01, 0.01, 0.01, 0.01, 0.11, 0.01, 0.01, 0.01],
            [0.70, 0.22, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]]

prob_seq = np.array(prob_seq)
beam_size = 3

result = beam_search_decoder(prob_seq, beam_size)

for seq, score in result:
    sentence = ""

    for word in seq:
        sentence += vocab[word] + " "

    print(sentence, "// Score: %.4f" % score)

커피 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 42.5243
커피 를 마셔 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 28.0135
마셔 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 17.8983


In [83]:
import torch
import torch.nn.functional as F

def calc_prob(src_ids, tgt_ids, model):
    # 마스크 생성 (PyTorch 버전)
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids,
        tgt_ids,
        enc_padding_mask,
        combined_mask,
        dec_padding_mask
    )

    # 마지막 차원에 대해 softmax 적용하여 확률값 계산
    return F.softmax(predictions, dim=-1)

print("슝=3")

슝=3


In [84]:
import numpy as np
import torch

def beam_search_decoder(sentence,
                        src_len,
                        tgt_len,
                        model,
                        src_tokenizer,
                        tgt_tokenizer,
                        beam_size):
    # 입력 문장을 토큰화
    tokens = src_tokenizer.encode_as_ids(sentence)

    # src_in: [1, src_len] 크기의 텐서로 padding (0: 패딩 토큰)
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    # beam search용 캐시 배열들
    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

    eos_flag = np.zeros((beam_size,), dtype=np.int64)  # EOS를 만난 branch 표시 (EOS: -1)
    scores = np.ones((beam_size,), dtype=np.float32)     # 각 branch의 score (확률 곱)

    # 디코더 첫 입력은 BOS 토큰
    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    # 초기 디코더 입력 (branch 0의 첫 토큰) -> shape: [1, 1]
    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    # calc_prob()는 softmax를 적용한 확률 텐서를 반환함
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    # seq_pos: 디코더 시퀀스 위치
    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        # 각 beam branch에 대해 캐시 초기화
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        # 각 beam branch에 대해 후보 확률 계산 및 캐시 업데이트
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                # 해당 branch의 현재까지의 시퀀스를 디코더 입력으로 변환
                dec_in_np = pred_cache[branch_idx, :seq_pos]
                dec_in = torch.tensor(dec_in_np, dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            # 각 branch 내에서 beam_size만큼의 후보 토큰을 선택
            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                # 후보 branch의 score 업데이트 (곱셈으로 누적)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                # 이미 선택된 토큰은 다시 선택되지 않도록 -1로 마킹
                prob[max_idx] = -1

        # 각 beam branch에서 최고 score를 가진 후보를 선택
        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1  # 해당 후보 제거

            # 만약 EOS 토큰이면 해당 branch는 종료 표시 (-1)
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    # 각 branch의 예측 시퀀스에서 EOS 토큰 이전까지만 추출하여 결과 반환
    pred = []
    for long_pred in pred_tmp:
        eos_token = tgt_tokenizer.eos_id()
        # EOS 토큰이 없는 경우, 전체 시퀀스를 사용하도록 처리할 수 있음
        try:
            eos_idx = list(long_pred).index(eos_token)
        except ValueError:
            eos_idx = tgt_len - 1
        short_pred = long_pred[:eos_idx+1]
        pred.append(short_pred.tolist())

    return pred

print("슝=3")

슝=3


In [85]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                            candidate,
                            weights=weights,
                            smoothing_function=SmoothingFunction().method1)

print('슝=3')

슝=3


In [86]:
def beam_bleu(reference, ids, tokenizer):
    # 기준 문장을 토큰화
    reference_tokens = reference.split()

    total_score = 0.0
    num_candidates = len(ids)
    if num_candidates == 0:
        return 0.0

    for candidate_ids in ids:
        # 후보 문장을 디코딩 후 토큰화
        candidate_sentence = tokenizer.decode_ids(candidate_ids)
        candidate_tokens = candidate_sentence.split()

        score = calculate_bleu(reference_tokens, candidate_tokens)

        print(f"Reference: {reference_tokens}")
        print(f"Candidate: {candidate_tokens}")
        print(f"BLEU: {score}")

        total_score += score

    return total_score / num_candidates

print("슝=3")

슝=3


In [97]:
# Q. 인덱스를 바꿔가며 확인해 보세요
test_idx = 8

ids = \
beam_search_decoder(test_kor_q_sentences[test_idx],
                    MAX_LEN,
                    MAX_LEN,
                    transformer,
                    tokenizer,
                    tokenizer,
                    beam_size=5)

bleu = beam_bleu(test_kor_a_sentences[test_idx], ids, tokenizer)
print(bleu)

Reference: ['음료수라도', '전하면서', '대화를', '나눠봐요', '.']
Candidate: ['춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요']
BLEU: 0
Reference: ['음료수라도', '전하면서', '대화를', '나눠봐요', '.']
Candidate: ['춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요', '.']
BLEU: 0.008687475782716616
Reference: ['음료수라도', '전하면서', '대화를', '나눠봐요', '.']
Candidate: ['춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고', '춥고네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요네요']
BLEU: 0
Reference: ['음료수라도', '전하면서', '대화를', '나눠봐요', '.']
Candidate: []
BLEU: 0
Reference: ['음료수라도', '전하면서', '대화를', '나눠봐요', '.']
Candidate: ['춥고']
BLEU: 0
0.0017374951565433232


In [88]:
!pip install gensim

In [31]:
# !pip uninstall gensim -y
# !pip install gensim==3.8.3 --use-pep517

In [32]:
# import gensim.downloader as api

# wv = api.load('glove-wiki-gigaword-300')

import urllib.request
from gensim.models import KeyedVectors

# 1. FastText 한국어 모델 다운로드 (약 800MB)
# url = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.ko.vec"
# urllib.request.urlretrieve(url, "wiki.ko.vec")

# 2. 로드 (Gensim 4.x에서 완벽 호환)
# print("FastText 모델 로드 중... (시간이 조금 걸릴 수 있습니다)")
# wv = KeyedVectors.load_word2vec_format("./ko.bin")
# print("로드 완료!")


# 앞서 작성한 lexical_sub_mecab 함수에 word2vec 대신 wv 객체를 넘겨주면 됩니다.


from gensim.models import KeyedVectors
wv = KeyedVectors.load_word2vec_format('./ko_vectors.txt')

In [33]:
wv.most_similar("바나나")

[('코코넛', 0.8097118735313416),
 ('시금치', 0.7701147198677063),
 ('레몬', 0.76884925365448),
 ('땅콩', 0.7684734463691711),
 ('파인애플', 0.7639914751052856),
 ('녹차', 0.7631460428237915),
 ('딸기', 0.7617197036743164),
 ('바닐라', 0.7497864961624146),
 ('파슬리', 0.7447543740272522),
 ('코코아', 0.7408245205879211)]

In [34]:
sample_sentence = "너 알아? 필요한 건 단지 어텐션이야 ."
sample_tokens = sample_sentence.split()

selected_tok = random.choice(sample_tokens)

result = ""
for tok in sample_tokens:
    if tok is selected_tok:
        result += wv.most_similar(tok)[0][0] + " "

    else:
        result += tok + " "

print("From:", sample_sentence)
print("To:", result)

From: 너 알아? 필요한 건 단지 어텐션이야 .
To: 너 알아? 필요한 건 단지 어텐션이야 는데 


In [35]:
# Q. Lexical Substitution 을 구현해봅시다.
def lexical_sub(sentence, wv):
    # 문장을 토큰화
    tokens = sentence.split()

    # 유효한 단어 필터링 (임베딩에 존재하는 단어만 고려)
    valid_tokens = [tok for tok in tokens if tok in wv]

    # 대체할 단어 선택 (임베딩 내 존재하는 단어 중 하나)
    if not valid_tokens:
        return sentence  # 모든 단어가 임베딩 내에 없으면 원래 문장 반환

    selected_tok = random.choice(valid_tokens)

    # 가장 유사한 단어 찾기
    similar_word = wv.most_similar(selected_tok)[0][0]

    # 변환된 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])

    return new_sentence

    import random
# from konlpy.tag import Mecab

# Mecab 객체 초기화
# mecab = Mecab()

def lexical_sub_mecab(sentence, wv):
    # 1. 띄어쓰기 단위로 어절 분리
    tokens = sentence.split()
    
    candidates = []
    
    # 2. 각 어절 단위로 형태소를 분석하여 유효한 교체 후보 찾기
    for i, token in enumerate(tokens):
        # 어절 단위 형태소 분석
        pos_tags = mecab.pos(token)
        
        for word, pos in pos_tags:
            # 명사(N...)나 동사/형용사(V...) 위주로 교체 (조사/어미 제외)
            if pos.startswith('N') or pos.startswith('V'):
                # 해당 형태소가 임베딩 모델(wv)에 존재하는지 확인
                if word in wv:
                    # (어절 인덱스, 원본 어절, 교체 대상 형태소) 형태로 저장
                    candidates.append((i, token, word))
    
    # 3. 대체할 단어가 없으면 원본 문장 반환
    if not candidates:
        return sentence
    
    # 4. 교체할 형태소 무작위 선택
    target_idx, original_token, target_morpheme = random.choice(candidates)
    
    # 5. Word2Vec에서 가장 유사한 단어 추출
    try:
        similar_word = wv.most_similar(target_morpheme)[0][0]
    except KeyError:
        return sentence
    
    # 6. 원본 어절에서 형태소 부분만 유사한 단어로 치환 (예: '영화가' -> '영화'를 '드라마'로 -> '드라마가')
    # replace(old, new, 1)을 사용하여 해당 형태소 1개만 치환
    new_token = original_token.replace(target_morpheme, similar_word, 1)
    
    # 7. 변환된 어절을 원래 위치에 넣고 문장 재구성
    tokens[target_idx] = new_token
    new_sentence = " ".join(tokens)
    
    return new_sentence

In [36]:
new_corpus = []

for old_src in tqdm(test_kor_q_sentences):
    new_src = lexical_sub_mecab(old_src, wv)
    if new_src is not None:
        new_corpus.append(new_src)
    # Augmentation이 없더라도 원본 문장을 포함시킵니다
    new_corpus.append(old_src)

print(new_corpus[:10])

100%|██████████| 59/59 [00:00<00:00, 510.53it/s]

['학교샘 좋아시키는 사람 있나 ?', '학교샘 좋아하는 사람 있나 ?', '학교에 심남 있는데 협조해볼까 ?', '학교에 심남 있는데 연락해볼까 ?', '학교에 좋아하는 여자애의 불만을 어떻게 끌지 ?', '학교에 좋아하는 여자애의 관심을 어떻게 끌지 ?', '학교에 좋아하는 오빠한테 이렇게 다가갈까 ?', '학교에 좋아하는 오빠한테 어떻게 다가갈까 ?', '학생일 때 썸 좋을까', '학생일 때 썸 괜찮을까']
